# Task 01 — Bronze Load

**Workshop: Final Pipeline | Stage 1 of 3**

## Goal

Read `orders_batch.json` from a Unity Catalog Volume and write it as a **managed Delta table** in the Bronze layer.

```
[Volume]  DATASET_PATH/orders/orders_batch.json
                    │
                    ▼  spark.read (manual schema)
                    │
                    ▼  .write.format("delta").saveAsTable(...)
              bronze.lab_orders
```

## Expected output

| Column | Type | Notes |
|--------|------|-------|
| order_id | StringType | Not null |
| customer_id | StringType | |
| order_datetime | StringType | Will be cast in Task 02 |
| quantity | IntegerType | |
| unit_price | DoubleType | |
| discount_percent | DoubleType | |
| total_amount | DoubleType | |
| payment_method | StringType | |

> Run cells **top to bottom**. Do not skip the widget setup.


In [ ]:
%run ../../setup/00_setup

In [ ]:
# ── Widgets ── (run as-is, values injected by the Job)
dbutils.widgets.text("source_path", DATASET_PATH, "Source Path")
dbutils.widgets.text("catalog",     CATALOG,       "Catalog")
dbutils.widgets.text("schema",      BRONZE_SCHEMA, "Bronze Schema")

source_path   = dbutils.widgets.get("source_path")
catalog       = dbutils.widgets.get("catalog")
schema        = dbutils.widgets.get("schema")

orders_path   = f"{source_path}/orders/orders_batch.json"
target_table  = f"{catalog}.{schema}.lab_orders"

print(f"Source : {orders_path}")
print(f"Target : {target_table}")


---

## Exercise 1 — Import types

Import `StructType`, `StructField` and all column types you will need for the schema below.


In [ ]:
from pyspark.sql.functions import current_timestamp

# 💡 HINT — imports:
#   from pyspark.sql.types import (
#       StructType, StructField,
#       StringType, IntegerType, DoubleType,   # ← pick what you need
#       TimestampType, DateType
#   )

# YOUR CODE HERE
raise NotImplementedError("Complete this cell before proceeding")

---

## Exercise 2 — Define the schema

Define a `StructType` with the 8 columns from the table above.  
Use `nullable=False` only for `order_id`.


In [ ]:
# 💡 HINT — StructType schema definition:
#
#   orders_schema = StructType([
#       StructField("column_name", DataType(), nullable=True),
#       ...
#   ])
#
#   DataType examples:
#       StringType()   IntegerType()   DoubleType()
#       DateType()     TimestampType()
#
#   nullable=False means Spark will reject null values in that column.

# YOUR CODE HERE
raise NotImplementedError("Complete this cell before proceeding")

orders_schema  # show the schema

---

## Exercise 3 — Read the JSON file

Use `spark.read` with the schema you defined above.  
Load from `orders_path` (already set in the widgets cell).


In [ ]:
# 💡 HINT — spark.read for JSON:
#
#   df = (
#       spark.read
#           .format("json")
#           .schema(your_schema)   # ← pass the StructType
#           .load(path)            # ← path string
#   )
#
#   Note: do NOT use inferSchema=True — use your explicit schema.

# YOUR CODE HERE
raise NotImplementedError("Complete this cell before proceeding")

In [ ]:
# Verify — do not modify
print(f"Rows loaded : {orders_raw.count():,}")
print()
orders_raw.printSchema()
display(orders_raw.limit(5))


---

## Exercise 4 — Write to Delta (managed table)

Write `orders_raw` as a managed Delta table named `lab_orders` in the Bronze schema.  
Use `mode("overwrite")` so re-running the notebook is safe.


In [ ]:
# 💡 HINT — DataFrame.write:
#
#   df.write
#       .format("delta")
#       .mode("overwrite")           # overwrite | append | errorIfExists
#       .option("overwriteSchema", "true")   # allow schema changes on overwrite
#       .saveAsTable("catalog.schema.table") # creates managed table in UC

# YOUR CODE HERE
raise NotImplementedError("Complete this cell before proceeding")

In [ ]:
# Verify — do not modify
row_count = spark.table(target_table).count()
print(f"✅ {target_table}  →  {row_count:,} rows")
spark.sql(f"DESCRIBE HISTORY {target_table}").select("version","timestamp","operation").show(3, truncate=False)


In [ ]:
# Exit — used by Job to pass result to next task
import json
dbutils.notebook.exit(json.dumps({
    "status": "SUCCESS",
    "table": target_table,
    "rows": row_count
}))
